# AI Agent Loop Demo


This notebook demonstrates the core mechanism behind an AI agent: a model that can decide to call a tool, receive the result, and continue reasoning toward a goal — all in a loop.

---

This notebook contains starter code. Please fill in details as needed.
---

## Setup - Gemini API key

How to get the key (takes about two minutes):

* Go to [aistudio.google.com](https://aistudio.google.com/prompts/new_chat)
* Sign in with your Google account
* Click "Get API key" in the left sidebar
* Click "Create API key" — choose "Create API key in new project" if prompted
* Copy the key that appears (it starts with AIza...)

> The free tier allows up to 10 requests per minute and 250 requests per day, which is enough to work through this notebook.

If you see occasional slow-downs when you run several questions back to back — that is normal, and the code handles it automatically.

## References
* [Google agent loop](https://github.com/xiaosuhu/midas-ai-in-research/blob/v1.0-dev/docs/notebooks/agent_loop_demo.ipynb)
* https://ai.google.dev/gemini-api/docs/thinking#signatures
* https://ai.google.dev/gemini-api/docs/text-generation
* [Developer guide](https://github.com/xiaosuhu/midas-ai-in-research/blob/v1.0-dev/docs/notebooks/agent_loop_demo.ipynb)

### Where to put the key
In Colab, open the **key icon (Secrets)** in the left sidebar, add a secret named `GOOGLE_API_KEY`, paste your key as the value, and switch on **Notebook access**. If no secret is found, the notebook will ask you to paste the key (hidden input). The key is never written into the notebook.


In [23]:
!pip install google-auth==2.49.0 "google-genai>=2.12.1,<3" --quiet
print("Done.")

Done.


In [24]:
import os

try:
    from google.colab import userdata
    os.environ["GOOGLE_API_KEY"] = userdata.get("GOOGLE_API_KEY")
    print("API key loaded from Colab Secrets.")
except Exception:
    import getpass
    os.environ["GOOGLE_API_KEY"] = getpass.getpass("Paste your Gemini API key: ")
    print("API key set.")


API key loaded from Colab Secrets.


In [25]:
# %% Cell 3: recipe database
DOCUMENTS = {
    "veg_paneer_burrito": (
        "For the paneer filling: 200g paneer, cut into small cubes. "
        "1 small onion, finely sliced. "
        "1 small capsicum, sliced. "
        "1 tomato, chopped. "
        "1 tbsp oil. "
        "1 tsp ginger-garlic paste. "
        "1/2 tsp cumin powder. "
        "1/2 tsp coriander powder. "
        "1/2 tsp red chili powder. "
        "1/4 tsp turmeric powder. "
        "Salt to taste. "
        "For the burrito: 2 large flour tortillas. "
        "1 cup cooked rice. "
        "1/2 cup cooked rajma or black beans, drained. "
        "1/2 cup shredded lettuce. "
        "1/2 cup grated cheese. "
        "1/4 cup salsa. "
        "2 tbsp sour cream or plain yogurt. "
        "1 tbsp chopped coriander leaves. "
        "Heat oil in a pan. Add onion and capsicum and saute until slightly soft. "
        "Add ginger-garlic paste and cook for 30 seconds. "
        "Add tomato, cumin powder, coriander powder, red chili powder, turmeric, and salt. "
        "Cook until the tomato softens and the masala thickens. "
        "Add paneer cubes and gently toss until coated with the masala. Cook for 3-4 minutes without breaking the paneer. "
        "Warm each tortilla on a pan until soft. "
        "Place cooked rice, beans, paneer filling, lettuce, cheese, salsa, and sour cream in a line near the center. "
        "Sprinkle chopped coriander leaves. Fold the sides inward, roll the tortilla tightly, and serve. "
        "For a crisp burrito, place the rolled burrito seam-side down on a hot pan and toast for 1-2 minutes on each side."
    ),

    "gudbud_ice_cream": (
        "Gudbud is a layered ice cream dessert popular in Mangaluru. "
        "For 2 servings: 2 scoops vanilla ice cream. "
        "2 scoops strawberry ice cream. "
        "2 scoops chocolate or butterscotch ice cream. "
        "1 small banana, sliced. "
        "1/4 cup chopped seasonal fruits such as apple, grapes, and mango. "
        "2 tbsp chopped jelly pieces. "
        "2 tbsp chopped roasted nuts such as cashews and almonds. "
        "2 tbsp tutti frutti. "
        "2 tbsp strawberry or rose syrup. "
        "2 tbsp chocolate syrup. "
        "2 tbsp sweetened milk or condensed milk, optional. "
        "2 tbsp whipped cream, optional. "
        "Use two tall dessert glasses. Add a little fruit and jelly to the bottom of each glass. "
        "Add a layer of vanilla ice cream, followed by sliced banana and some chopped nuts. "
        "Add a layer of strawberry ice cream and a little strawberry or rose syrup. "
        "Add another layer of chopped fruit, jelly, and tutti frutti. "
        "Top with chocolate or butterscotch ice cream. "
        "Drizzle chocolate syrup and optional sweetened milk over the layers. "
        "Finish with whipped cream, chopped nuts, and tutti frutti. "
        "Serve immediately while the ice cream is still frozen. "
        "The layers can be adjusted according to taste; keep the fruit and toppings ready before assembling."
    ),

    "veggie_farmhouse_pizza": (
        "For one 10-12 inch pizza: 1 prepared pizza base or 250g pizza dough. "
        "1/3 cup pizza sauce. "
        "1 cup grated mozzarella cheese. "
        "1/2 small capsicum, thinly sliced. "
        "1 small onion, thinly sliced. "
        "1 small tomato, deseeded and sliced. "
        "4-5 button mushrooms, sliced. "
        "1/4 cup sweet corn kernels. "
        "4-5 black olives, sliced, optional. "
        "1/2 tsp mixed Italian herbs. "
        "1/4 tsp chili flakes, optional. "
        "1 tsp olive oil. "
        "Salt and black pepper to taste. "
        "If using homemade dough, allow it to rise until doubled in size, then shape it into a 10-12 inch round base. "
        "Preheat the oven to 220 degrees Celsius. "
        "Place the pizza base on a baking tray or pizza stone. "
        "Spread pizza sauce evenly over the base, leaving a small border around the edge. "
        "Sprinkle half of the mozzarella cheese over the sauce. "
        "Arrange capsicum, onion, tomato, mushrooms, sweet corn, and olives evenly over the cheese. "
        "Season lightly with salt, black pepper, Italian herbs, and chili flakes. "
        "Add the remaining mozzarella cheese and drizzle olive oil over the toppings. "
        "Bake for 10-15 minutes, or until the cheese melts and the crust becomes golden. "
        "Cooking time may vary depending on the oven and thickness of the base. "
        "Allow the pizza to rest for 2 minutes, slice, and serve hot."
    ),

    "masala_dosa": (
        "Masala dosa is a crisp South Indian crepe filled with spiced potato. "
        "For 4 dosas: 1 cup dosa batter, fermented. "
        "3 medium potatoes, boiled and roughly mashed. "
        "1 medium onion, thinly sliced. "
        "1 tbsp oil, plus extra for cooking the dosas. "
        "1/2 tsp mustard seeds. "
        "1 sprig curry leaves. "
        "1 green chili, chopped. "
        "1/2 tsp ginger, grated. "
        "1/4 tsp turmeric powder. "
        "Salt to taste. "
        "Heat oil in a pan and add the mustard seeds until they splutter. "
        "Add curry leaves, green chili, ginger, and onion, and saute until the onion turns soft and translucent. "
        "Add turmeric and salt, then mix in the mashed potatoes and cook for 2-3 minutes. "
        "Heat a flat griddle or non-stick pan until hot and grease it lightly. "
        "Pour a ladle of batter in the center and spread it outward in a circle to make a thin crepe. "
        "Drizzle a little oil around the edges and cook for 2-3 minutes, until the underside is golden and crisp. "
        "Place a portion of the potato filling in the center, fold the dosa over, and serve hot with coconut chutney and sambar."
    )
}

In [26]:
import re

STOPWORDS = {
    "a", "an", "the", "and", "or", "of", "to", "in", "on", "for", "with", "is",
    "are", "how", "what", "which", "do", "does", "can", "i", "we", "our", "it",
    "long", "much", "many", "need", "needed", "make", "ingredients", "recipe",
}


def search_documents(query: str) -> str:
    """Search the cookbook for recipes relevant to the query."""
    words = {w for w in re.findall(r"[a-z]+", query.lower()) if w not in STOPWORDS}
    results = []

    for name, content in DOCUMENTS.items():
        title_words = set(name.replace("_", " ").split())
        text_words = set(re.findall(r"[a-z]+", content.lower()))
        score = sum(3 for w in words if w in title_words)   # title match counts more
        score += sum(1 for w in words if w in text_words)   # whole-word content match
        if score > 0:
            results.append((score, name, content))

    results.sort(key=lambda r: r[0], reverse=True)

    if not results:
        return "No relevant recipes found for that query."

    return "\n\n".join(f"[Source: {n}]\n{c}" for _, n, c in results[:2])


print(search_documents("gudbud_ice_cream"))


[Source: gudbud_ice_cream]
Gudbud is a layered ice cream dessert popular in Mangaluru. For 2 servings: 2 scoops vanilla ice cream. 2 scoops strawberry ice cream. 2 scoops chocolate or butterscotch ice cream. 1 small banana, sliced. 1/4 cup chopped seasonal fruits such as apple, grapes, and mango. 2 tbsp chopped jelly pieces. 2 tbsp chopped roasted nuts such as cashews and almonds. 2 tbsp tutti frutti. 2 tbsp strawberry or rose syrup. 2 tbsp chocolate syrup. 2 tbsp sweetened milk or condensed milk, optional. 2 tbsp whipped cream, optional. Use two tall dessert glasses. Add a little fruit and jelly to the bottom of each glass. Add a layer of vanilla ice cream, followed by sliced banana and some chopped nuts. Add a layer of strawberry ice cream and a little strawberry or rose syrup. Add another layer of chopped fruit, jelly, and tutti frutti. Top with chocolate or butterscotch ice cream. Drizzle chocolate syrup and optional sweetened milk over the layers. Finish with whipped cream, choppe

In [27]:
import time
from google import genai
from google.genai import types

client = genai.Client(api_key=os.environ["GOOGLE_API_KEY"])

# NOTE: gemini-2.5-flash returns 404 for new users; gemini-3.8-flash is the current model.
# To list models available to your key:
#   for m in client.models.list(): print(m.name)
MODEL_CANDIDATES = [
    "gemini-2.5-flash",
    "gemini-3.5-flash-lite"
] # tried in order

search_tool = types.Tool(
    function_declarations=[
        types.FunctionDeclaration(
            name="search_documents",
            description="Search the Multi-Cuisine Recipe Collection for relevant recipes and answer the user's questions accurately using only the information available in the provided documents.",
            parameters={
                "type": "object",
                "properties": {"query": {"type": "string"}},
                "required": ["query"],
            },
        )
    ]
)

config = types.GenerateContentConfig(
    system_instruction=(
        "You are a professional multicuisine vegetarian chef specializing in "
"Mexican, Indian, and Italian-inspired dishes. "
"Use search_documents for questions about ingredients, recipes, cooking "
"times, and preparation methods. Answer only using the retrieved documents. "
"If the documents do not contain the answer, say so clearly."
    ),
    tools=[search_tool],
    automatic_function_calling=types.AutomaticFunctionCallingConfig(disable=True),
)

TOOLS = {"search_documents": search_documents}


In [28]:
# Free tier allows 10 requests/minute. Instead of sleeping a fixed time between
# questions, track recent calls and only wait when we are about to exceed the limit.
MAX_CALLS_PER_MIN = 9          # 1 below the limit, as a safety margin
_call_times = []

def wait_for_rate_limit():
    now = time.time()
    while _call_times and now - _call_times[0] > 60:
        _call_times.pop(0)
    if len(_call_times) >= MAX_CALLS_PER_MIN:
        wait = 60 - (now - _call_times[0]) + 0.5
        print(f"  Pausing {wait:.0f}s to stay within the free-tier rate limit...")
        time.sleep(wait)
    _call_times.append(time.time())


def send_with_retry(contents, max_retries=3):
    last_error = None
    for model in MODEL_CANDIDATES:
        for attempt in range(max_retries):
            try:
                wait_for_rate_limit()
                return client.models.generate_content(
                    model=model, contents=contents, config=config
                )
            except Exception as e:
                err = str(e)
                last_error = e
                if "404" in err:                       # model not available -> next model
                    break
                if not any(c in err for c in ("429", "500", "502", "503", "504")):
                    raise                              # a real bug, don't hide it
                # 429 = per-minute quota (needs a longer pause); 5xx = server busy (short pause is enough)
                wait = (15 if "429" in err else 5) * (attempt + 1)
                print(f"{model}: temporary error ({err[:120]}). Retrying in {wait}s...")
                time.sleep(wait)
        print(f"Switching away from {model}...")
    raise last_error


def run_agent(question: str, max_steps: int = 8) -> str:
    print(f"Question: {question}\n" + "-" * 50)
    contents = [types.Content(role="user", parts=[types.Part(text=question)])]

    for step in range(1, max_steps + 1):
        response = send_with_retry(contents)

        if not response.candidates or not response.candidates[0].content:
            print("No response returned by Gemini.")
            return ""

        model_content = response.candidates[0].content
        contents.append(model_content)   # keep the model turn unmodified

        calls = [p.function_call for p in (model_content.parts or []) if p.function_call]

        if not calls:
            answer = response.text or "No text answer returned."
            print(f"\nFinal answer:\n{answer}")
            return answer

        result_parts = []
        for call in calls:
            func = TOOLS.get(call.name)
            if func is None:
                result = "Unknown tool."
            else:
                args = dict(call.args)
                print(f"Step {step}: {call.name}({args})")
                result = func(**args)
                print(f"  Result preview: {result[:100]}...")

            result_parts.append(
                types.Part(
                    function_response=types.FunctionResponse(
                        id=call.id,          # may be None; that is fine
                        name=call.name,
                        response={"result": result},
                    )
                )
            )

        contents.append(types.Content(role="user", parts=result_parts))

    print("Stopped: maximum steps reached.")
    return ""


In [29]:
questions = [
    "What ingredients are needed to make a veg paneer burrito?",
    "How do I assemble Gudbud ice cream, and what layers does it contain?",
    "What vegetables and baking temperature are used for a veggie farmhouse pizza?",
    "Which of these recipes use cheese, and how much does each one need?",   # needs info from more than one recipe
    "How do I make chicken biryani?",   # NOT in the documents -> the agent should say it can't find it
    "What is the most popular food item in Canada?",   # general knowledge -> the agent should not answer from its own knowledge
    "What do I serve with masala dosa, and how long do I cook each dosa?",   # answerable from the new 4th document
]

for q in questions:
    # Rate limiting is handled inside send_with_retry (no fixed sleep needed).
    print("\n" + "=" * 60)
    run_agent(q)


Question: What ingredients are needed to make a veg paneer burrito?
--------------------------------------------------
Switching away from gemini-2.5-flash...
Step 1: search_documents({'query': 'veg paneer burrito'})
  Result preview: [Source: veg_paneer_burrito]
For the paneer filling: 200g paneer, cut into small cubes. 1 small onio...
Switching away from gemini-2.5-flash...

Final answer:
To make a veg paneer burrito, you will need the following ingredients:

**For the paneer filling:**
* 200g paneer, cut into small cubes
* 1 small onion, finely sliced
* 1 small capsicum, sliced
* 1 tomato, chopped
* 1 tbsp oil
* 1 tsp ginger-garlic paste
* 1/2 tsp cumin powder
* 1/2 tsp coriander powder
* 1/2 tsp red chili powder
* 1/4 tsp turmeric powder
* Salt to taste

**For the burrito:**
* 2 large flour tortillas
* 1 cup cooked rice
* 1/2 cup cooked rajma or black beans, drained
* 1/2 cup shredded lettuce
* 1/2 cup grated cheese
* 1/4 cup salsa
* 2 tbsp sour cream or plain yogurt
* 1 tbsp chop

In [31]:
# This cell will check if you have added atleast 2 more questions.
assert(len(questions) >= 4)
# This will check if you added a question.
assert("Add a question here." not in questions)